In [163]:
#Importing files
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore') #fils are deprected and we are in development phase

In [164]:
df = sns.load_dataset('titanic')
df

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0.0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0.0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0.0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0.0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0.0,8.0500,S,Third,man,True,NaN,Southampton,no,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
378,0,3,male,20.0,0,0.0,4.0125,C,Third,man,True,NaN,Cherbourg,no,True
379,0,3,male,19.0,0,0.0,7.7750,S,Third,man,True,NaN,Southampton,no,True
380,1,1,female,42.0,0,0.0,227.5250,C,First,woman,False,NaN,Cherbourg,yes,True
381,1,3,female,1.0,0,2.0,15.7417,C,Third,child,False,NaN,Cherbourg,yes,False


In [165]:
df.shape

(383, 15)

In [166]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 383 entries, 0 to 382
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   survived     383 non-null    int64   
 1   pclass       383 non-null    int64   
 2   sex          383 non-null    object  
 3   age          307 non-null    float64 
 4   sibsp        383 non-null    int64   
 5   parch        382 non-null    float64 
 6   fare         382 non-null    float64 
 7   embarked     381 non-null    object  
 8   class        382 non-null    category
 9   who          382 non-null    object  
 10  adult_male   382 non-null    object  
 11  deck         87 non-null     category
 12  embark_town  381 non-null    object  
 13  alive        382 non-null    object  
 14  alone        382 non-null    object  
dtypes: category(2), float64(3), int64(3), object(7)
memory usage: 40.3+ KB


In [167]:
df.columns

Index(['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare',
       'embarked', 'class', 'who', 'adult_male', 'deck', 'embark_town',
       'alive', 'alone'],
      dtype='object')

In [168]:
df.drop(['deck','embark_town','alive','class','who','adult_male'],axis=1,inplace=True)

In [169]:
df['age'].fillna(df['age'].mean(),inplace=True)

In [170]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 383 entries, 0 to 382
Data columns (total 9 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   survived  383 non-null    int64  
 1   pclass    383 non-null    int64  
 2   sex       383 non-null    object 
 3   age       383 non-null    float64
 4   sibsp     383 non-null    int64  
 5   parch     382 non-null    float64
 6   fare      382 non-null    float64
 7   embarked  381 non-null    object 
 8   alone     382 non-null    object 
dtypes: float64(3), int64(3), object(3)
memory usage: 27.1+ KB


In [171]:
# only removing null row from the embarked column.
df.dropna(subset=["embarked"], inplace=True)

In [172]:
# label encode
from sklearn.preprocessing import LabelEncoder

In [173]:
le = LabelEncoder()
df['sex'] = le.fit_transform(df['sex'])
df['embarked'] = le.fit_transform(df['embarked'])

In [174]:
df = df.astype(int)

In [175]:
df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,alone
0,0,3,1,22,1,0,7,2,0
1,1,1,0,38,1,0,71,0,0
2,1,3,0,26,0,0,7,2,1
3,1,1,0,35,1,0,53,2,0
4,0,3,1,35,0,0,8,2,1


In [176]:
X = df.drop("survived",axis=1)
y = df['survived']

In [177]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

In [178]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [179]:
models = {
    "Logistic Regression": LogisticRegression(),
    "KNN":KNeighborsClassifier(),
    "Naive Byeas":GaussianNB(),
    "SVM":SVC(),
    "Decision tree":DecisionTreeClassifier()
}

In [180]:
results = []

for name, model in models.items():
    model.fit(X_train_scaled, y_train)

    y_pred = model.predict(X_test_scaled)

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    results.append({
        "Model": name,
        "Accuracy": acc,
        "F1 Score": f1
    })

In [181]:
results_df = pd.DataFrame(results)
results_df.sort_values(by="Accuracy", ascending=False)

,Model,Accuracy,F1 Score
3,SVM,0.844156,0.800000
0,Logistic Regression,0.818182,0.766667
2,Naive Byeas,0.805195,0.754098
4,Decision tree,0.779221,0.730159
1,KNN,0.740260,0.677419


In [182]:
for name, model in models.items():
    scores = cross_val_score(model, X, y, cv=5)

    print(f"{name} CV Accuracy: {scores.mean():.4f}")

Logistic Regression CV Accuracy: 0.7953
KNN CV Accuracy: 0.6563
Naive Byeas CV Accuracy: 0.8058
SVM CV Accuracy: 0.6379
Decision tree CV Accuracy: 0.7535


In [183]:
accuracy_score(y_test,y_pred)

0.7792207792207793

In [184]:
confusion_matrix(y_test,y_pred)

array([[37, 10],
       [ 7, 23]])

In [185]:
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.84      0.79      0.81        47
           1       0.70      0.77      0.73        30

    accuracy                           0.78        77
   macro avg       0.77      0.78      0.77        77
weighted avg       0.78      0.78      0.78        77

